# LLM Magic Quadrant — Use-Case Recommender

**How it works:**
1. You describe your **use case** and provide **prompts** that represent real tasks
2. The notebook runs those prompts against all available models on your Databricks instance
3. Each model is scored on **Accuracy · Cost · Speed**
4. A **Magic Quadrant** is generated — showing where each model sits and which one is recommended

**Quadrant axes:**
- X → Performance Score (accuracy, higher = better)
- Y → Value Score (cost efficiency, higher = cheaper per token)
- Bubble size → Speed (larger = faster response)

**Recommendation weights** are tunable so the winner matches your priorities (quality vs cost vs speed).

---
## Cell 1 · Install & imports

In [ ]:
%pip install openai matplotlib numpy -q

In [ ]:
import os, re, ast, json, time, ssl, math
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import defaultdict
import openai

print("✓ imports ready")

---
## Cell 2 · Credentials

In [ ]:
DATABRICKS_HOST  = "https://dbc-7f2cefe3-e622.cloud.databricks.com"  # ← your workspace
DATABRICKS_TOKEN = "<YOUR-DATABRICKS-PAT>"              # ← your PAT

client = openai.OpenAI(
    base_url=f"{DATABRICKS_HOST.rstrip('/')}/serving-endpoints",
    api_key=DATABRICKS_TOKEN,
)
print("✓ client ready →", DATABRICKS_HOST)

---
## Cell 3 · Model registry

In [ ]:
# Add / remove models to match what your workspace has enabled
MODELS = [
    {"model_id": "databricks-claude-opus-4-7",             "display_name": "Claude Opus 4.7",        "provider": "Anthropic", "input_price_per_1m": 15.00, "output_price_per_1m": 75.00},
    {"model_id": "databricks-claude-sonnet-4-6",           "display_name": "Claude Sonnet 4.6",      "provider": "Anthropic", "input_price_per_1m":  3.00, "output_price_per_1m": 15.00},
    {"model_id": "databricks-claude-haiku-4-5",            "display_name": "Claude Haiku 4.5",       "provider": "Anthropic", "input_price_per_1m":  0.80, "output_price_per_1m":  4.00},
    {"model_id": "databricks-gpt-4o",                      "display_name": "GPT-4o",                 "provider": "OpenAI",    "input_price_per_1m":  2.50, "output_price_per_1m": 10.00},
    {"model_id": "databricks-gpt-4o-mini",                 "display_name": "GPT-4o-mini",            "provider": "OpenAI",    "input_price_per_1m":  0.15, "output_price_per_1m":  0.60},
    {"model_id": "databricks-gemini-2-5-pro",              "display_name": "Gemini 2.5 Pro",         "provider": "Google",    "input_price_per_1m":  1.25, "output_price_per_1m": 10.00},
    {"model_id": "databricks-gemini-2-5-flash",            "display_name": "Gemini 2.5 Flash",       "provider": "Google",    "input_price_per_1m":  0.075,"output_price_per_1m":  0.30},
    {"model_id": "databricks-llama-4-maverick",            "display_name": "Llama 4 Maverick",       "provider": "Meta",      "input_price_per_1m":  0.15, "output_price_per_1m":  0.60},
    {"model_id": "databricks-meta-llama-3-3-70b-instruct", "display_name": "Llama 3.3 70B",          "provider": "Meta",      "input_price_per_1m":  0.12, "output_price_per_1m":  0.38},
    {"model_id": "databricks-meta-llama-3-1-8b-instruct",  "display_name": "Llama 3.1 8B",           "provider": "Meta",      "input_price_per_1m":  0.02, "output_price_per_1m":  0.05},
]
PRICING = {m["model_id"]: m for m in MODELS}
print(f"✓ {len(MODELS)} models registered")

---
## Cell 4 · ⚙️ Define your use case  ← EDIT THIS

Replace the example below with your actual use case and prompts.  
Each prompt needs a `text` (what you send) and a `validator` (how to score the answer).  
Use the built-in validators or write a simple `lambda r: (label, score, detail)` for custom checks.

In [ ]:
# ── Validator helpers ─────────────────────────────────────────────────────────
def _tier(s): return "Excellent" if s>=1 else "Good" if s>=0.75 else "Partial" if s>=0.40 else "Poor"

def contains_all(*keywords):
    """All keywords must appear in response (case-insensitive)."""
    def v(r):
        t = r.lower()
        missing = [k for k in keywords if k.lower() not in t]
        score = 1 - len(missing)/len(keywords)
        return _tier(score), round(score,2), ("All keywords found" if not missing else f"Missing: {missing}")
    return v

def exact_number(expected, tolerance=0):
    """Response must contain the expected number."""
    def v(r):
        nums = re.findall(r"-?\d+(?:\.\d+)?", r.replace(",",""))
        if not nums: return "Poor", 0.0, "No number found"
        got = float(nums[0])
        diff = abs(got - expected)
        if diff <= max(tolerance, 0.01): return "Excellent", 1.0, f"Correct: {got}"
        if diff <= expected*0.1:         return "Partial",   0.4,  f"Close: {got} vs {expected}"
        return "Poor", 0.0, f"Wrong: {got} vs {expected}"
    return v

def valid_json(*required_keys):
    """Response must be valid JSON containing all required keys."""
    def v(r):
        try:
            m = re.search(r"(\{.*\}|\[.*\])", r, re.DOTALL)
            data = json.loads(m.group(1) if m else r.strip())
            if isinstance(data, list): data = data[0] if data else {}
            missing = [k for k in required_keys if k not in data]
            score = 1 - len(missing)/len(required_keys) if required_keys else 1.0
            return _tier(score), round(score,2), ("Valid JSON" if not missing else f"Missing keys: {missing}")
        except Exception as e:
            return "Poor", 0.0, f"Invalid JSON: {e}"
    return v

def run_python(*test_cases):
    """Extract Python from response, run test_cases: each is (call_string, expected_value)."""
    def v(r):
        m = re.search(r"```(?:python)?\s*\n(.*?)```", r, re.DOTALL)
        code = m.group(1).strip() if m else r.strip()
        ns = {}
        try: exec(compile(ast.parse(code),"<llm>","exec"), ns)
        except Exception as e: return "Poor", 0.0, f"Compile error: {e}"
        passed, fails = 0, []
        for call, exp in test_cases:
            try:
                res = eval(call, ns)
                if res == exp: passed += 1
                else: fails.append(f"{call}→{res!r}(exp {exp!r})")
            except Exception as e: fails.append(f"{call}→ERR:{e}")
        score = passed/len(test_cases)
        return _tier(score), round(score,2), f"{passed}/{len(test_cases)} passed" + (f": {'; '.join(fails)}" if fails else "")
    return v

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# ✏️  EDIT HERE — define your use case
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

USE_CASE_NAME = "Customer Support Automation"
USE_CASE_DESC = "Classify support tickets, generate replies, and extract structured data from customer messages."

USE_CASE_PROMPTS = [
    {
        "id": "ticket_classification",
        "prompt": (
            "Classify this support ticket into one category: Billing, Technical, Shipping, Returns, Other.\n"
            "Ticket: 'My order arrived damaged and I want a refund.'\n"
            "Reply with one word only."
        ),
        "validator": contains_all("returns"),
    },
    {
        "id": "sentiment_urgency",
        "prompt": (
            "Rate the urgency of this customer message as Low, Medium, or High, and its sentiment as Positive, Neutral, or Negative.\n"
            "Message: 'This is the third time my payment has failed! I need this fixed NOW or I am cancelling!'\n"
            "Format: 'Urgency: <level>, Sentiment: <value>'"
        ),
        "validator": contains_all("high", "negative"),
    },
    {
        "id": "reply_generation",
        "prompt": (
            "Write a professional customer support reply to this message. Keep it under 60 words.\n"
            "Message: 'Where is my order? It was supposed to arrive 3 days ago.'"
        ),
        "validator": contains_all("apologize", "order"),
    },
    {
        "id": "extract_customer_info",
        "prompt": (
            "Extract customer details from this message and return as JSON with fields: name, issue, product.\n"
            "Message: 'Hi, I am Sarah. My laptop model XZ-500 keeps crashing after the last update.'\n"
            "Return only the JSON."
        ),
        "validator": valid_json("name", "issue", "product"),
    },
    {
        "id": "faq_answer",
        "prompt": (
            "Answer this customer FAQ in 1-2 sentences:\n"
            "Q: How long does a refund take to process?"
        ),
        "validator": contains_all("refund", "business days"),
    },
    {
        "id": "summarise_thread",
        "prompt": (
            "Summarise this support thread in one sentence:\n"
            "Customer: My internet is down. Agent: Have you restarted the router? "
            "Customer: Yes, still down. Agent: I will escalate to our network team."
        ),
        "validator": contains_all("internet", "escalat"),
    },
]

print(f"✓ Use case: '{USE_CASE_NAME}'")
print(f"  {len(USE_CASE_PROMPTS)} prompts defined")
for p in USE_CASE_PROMPTS:
    print(f"  · {p['id']}")

---
## Cell 5 · ⚖️ Set recommendation weights  ← TUNE FOR YOUR PRIORITIES

In [ ]:
# Preset profiles — uncomment one or set custom weights below
# PROFILE = "balanced"          # equal weight to all three
# PROFILE = "quality_first"     # accuracy matters most
# PROFILE = "cost_sensitive"    # cost matters most
# PROFILE = "latency_critical"  # speed matters most
PROFILE = "balanced"

WEIGHT_PRESETS = {
    "balanced":         {"accuracy": 0.40, "cost": 0.30, "speed": 0.30},
    "quality_first":    {"accuracy": 0.60, "cost": 0.20, "speed": 0.20},
    "cost_sensitive":   {"accuracy": 0.30, "cost": 0.55, "speed": 0.15},
    "latency_critical": {"accuracy": 0.30, "cost": 0.20, "speed": 0.50},
}

WEIGHTS = WEIGHT_PRESETS[PROFILE]
# OR override manually:
# WEIGHTS = {"accuracy": 0.5, "cost": 0.3, "speed": 0.2}

assert abs(sum(WEIGHTS.values()) - 1.0) < 0.001, "Weights must sum to 1.0"

print(f"✓ Profile : {PROFILE}")
print(f"  Accuracy weight : {WEIGHTS['accuracy']:.0%}")
print(f"  Cost     weight : {WEIGHTS['cost']:.0%}")
print(f"  Speed    weight : {WEIGHTS['speed']:.0%}")

---
## Cell 6 · Run benchmark against your use-case prompts

In [ ]:
def calc_cost(model_id, inp, out):
    p = PRICING.get(model_id)
    if not p: return 0.0
    return (inp/1_000_000)*p["input_price_per_1m"] + (out/1_000_000)*p["output_price_per_1m"]

def call_with_retry(model_id, prompt_text, max_tokens=512, max_retries=4, base_wait=2):
    for attempt in range(max_retries):
        try:
            return client.chat.completions.create(
                model=model_id,
                messages=[{"role": "user", "content": prompt_text}],
                max_tokens=max_tokens,
                temperature=0,
            )
        except openai.RateLimitError:
            wait = base_wait * (2 ** attempt)
            if attempt < max_retries - 1:
                print(f"  ⏳ rate limit — retrying in {wait}s ...", flush=True)
                time.sleep(wait)
            else:
                raise

CALL_DELAY = 1.0   # seconds between calls — raise if you hit 429s
raw_results = []
total = len(MODELS) * len(USE_CASE_PROMPTS)
count = 0

for model in MODELS:
    for prompt in USE_CASE_PROMPTS:
        count += 1
        tag = f"[{count:>2}/{total}] {model['display_name']:<24} ← {prompt['id']}"
        print(tag, end="  ", flush=True)
        start = time.perf_counter()
        try:
            resp = call_with_retry(model["model_id"], prompt["prompt"])
            elapsed = round(time.perf_counter()-start, 3)
            text    = resp.choices[0].message.content or ""
            inp     = resp.usage.prompt_tokens     if resp.usage else 0
            out     = resp.usage.completion_tokens if resp.usage else 0
            cost    = calc_cost(model["model_id"], inp, out)
            label, score, detail = prompt["validator"](text)
            print(f"→ {label:<10} {score:.2f}  {elapsed:.2f}s  ${cost:.6f}")
            raw_results.append({"model": model["display_name"], "model_id": model["model_id"],
                "provider": model["provider"], "prompt_id": prompt["id"],
                "input_tokens": inp, "output_tokens": out, "total_tokens": inp+out,
                "ai_cost_usd": cost, "response_time_s": elapsed,
                "accuracy_label": label, "accuracy_score": score,
                "accuracy_detail": detail, "response": text, "error": None})
        except Exception as e:
            elapsed = round(time.perf_counter()-start, 3)
            print(f"→ SKIP (not available)")
            raw_results.append({"model": model["display_name"], "model_id": model["model_id"],
                "provider": model["provider"], "prompt_id": prompt["id"],
                "input_tokens": 0, "output_tokens": 0, "total_tokens": 0,
                "ai_cost_usd": 0, "response_time_s": elapsed,
                "accuracy_label": "Error", "accuracy_score": 0,
                "accuracy_detail": str(e), "response": "", "error": str(e)})
        time.sleep(CALL_DELAY)

print(f"\n✓ Done — {len(raw_results)} results  |  errors: {sum(1 for r in raw_results if r['error'])}")

---
## Cell 7 · Score computation

In [ ]:
from collections import defaultdict

# Aggregate per model — skip models with all errors
agg = defaultdict(lambda: {"acc":[], "time":[], "cost":[], "provider":"", "errors":0, "calls":0})
for r in raw_results:
    a = agg[r["model"]]
    a["provider"] = r["provider"]
    a["calls"] += 1
    if r["error"]:
        a["errors"] += 1
    else:
        a["acc"].append(r["accuracy_score"])
        a["time"].append(r["response_time_s"])
        a["cost"].append(r["ai_cost_usd"])

# Build model metrics — only models with at least one successful call
metrics = {}
for name, a in agg.items():
    if not a["acc"]: continue   # all failed — skip from quadrant
    metrics[name] = {
        "provider":    a["provider"],
        "avg_accuracy": sum(a["acc"]) / len(a["acc"]),
        "avg_time_s":   sum(a["time"]) / len(a["time"]),
        "total_cost":   sum(a["cost"]),
        "errors":       a["errors"],
        "calls":        a["calls"],
    }

model_names = list(metrics.keys())

# Normalise 0-1 across all available models
def normalise(vals):
    mn, mx = min(vals), max(vals)
    return [0.5 if mx==mn else (v-mn)/(mx-mn) for v in vals]

accs   = [metrics[m]["avg_accuracy"] for m in model_names]
times  = [metrics[m]["avg_time_s"]   for m in model_names]
costs  = [metrics[m]["total_cost"]   for m in model_names]

norm_acc   = normalise(accs)
norm_speed = normalise([1/t if t>0 else 0 for t in times])   # invert: fast = high score
norm_value = normalise([1/(c+1e-9) for c in costs])           # invert: cheap = high score

# Composite recommendation score
for i, m in enumerate(model_names):
    metrics[m]["norm_accuracy"] = norm_acc[i]
    metrics[m]["norm_speed"]    = norm_speed[i]
    metrics[m]["norm_value"]    = norm_value[i]
    metrics[m]["composite"]     = (
        WEIGHTS["accuracy"] * norm_acc[i]
        + WEIGHTS["cost"]    * norm_value[i]
        + WEIGHTS["speed"]   * norm_speed[i]
    )

# Rank
ranked = sorted(model_names, key=lambda m: metrics[m]["composite"], reverse=True)
winner = ranked[0]

print(f"{'Model':<28} {'Accuracy':>9} {'Avg Time':>10} {'Total Cost':>12} {'Composite':>10}")
print("─"*75)
for m in ranked:
    x = metrics[m]
    medal = " ⭐ RECOMMENDED" if m==winner else ""
    print(f"{m:<28} {x['avg_accuracy']:>9.2f} {x['avg_time_s']:>9.2f}s ${x['total_cost']:>10.6f} {x['composite']:>10.3f}{medal}")

---
## Cell 8 · Magic Quadrant

In [ ]:
PROVIDER_COLORS = {
    "Anthropic": "#E07B39",
    "OpenAI":    "#10A37F",
    "Google":    "#4285F4",
    "Meta":      "#1877F2",
}

fig, axes = plt.subplots(1, 2, figsize=(20, 9),
                         gridspec_kw={"width_ratios": [2, 1]})
fig.patch.set_facecolor("#F4F6F9")
fig.suptitle(
    f"LLM Magic Quadrant — {USE_CASE_NAME}\n"
    f"Profile: {PROFILE.replace('_',' ').title()}  "
    f"(Accuracy {WEIGHTS['accuracy']:.0%}  ·  Cost {WEIGHTS['cost']:.0%}  ·  Speed {WEIGHTS['speed']:.0%})",
    fontsize=15, fontweight="bold", y=1.01
)

# ── Left panel: Magic Quadrant scatter ───────────────────────────────────────
ax = axes[0]
ax.set_facecolor("#FFFFFF")
ax.set_xlim(-0.05, 1.15)
ax.set_ylim(-0.05, 1.15)
ax.set_xlabel("Performance Score  (Accuracy →)", fontsize=12, labelpad=8)
ax.set_ylabel("Value Score  (Cost Efficiency →)", fontsize=12, labelpad=8)
ax.set_title("Magic Quadrant", fontsize=13, fontweight="bold", pad=10)

# Quadrant dividers at 0.5
ax.axvline(0.5, color="#BBBBBB", lw=1.5, ls="--", zorder=1)
ax.axhline(0.5, color="#BBBBBB", lw=1.5, ls="--", zorder=1)

# Quadrant background shading
quad_cfg = [
    (0.5, 0.5, 0.65, 0.65, "#E8F5E9", "⭐ LEADERS\n(Recommended)",   "#2e7d32", "center"),
    (0.0, 0.5, 0.50, 0.65, "#FFF3E0", "💰 BUDGET OPTIONS\n(Lower quality)",  "#e65100", "center"),
    (0.5, 0.0, 0.65, 0.50, "#E3F2FD", "🏆 PREMIUM QUALITY\n(Higher cost)",   "#1565c0", "center"),
    (0.0, 0.0, 0.50, 0.50, "#FFEBEE", "⚠️  AVOID\n(Costly & inaccurate)",  "#b71c1c", "center"),
]
for x0,y0,lx,ly,bg,label,tc,ha in quad_cfg:
    ax.add_patch(plt.Rectangle((x0,y0), 0.5, 0.5, color=bg, zorder=0, alpha=0.6))
    ax.text(x0+0.25, y0+0.25, label, ha="center", va="center",
            fontsize=8.5, color=tc, alpha=0.55, fontweight="bold", linespacing=1.5)

# Plot each model
used_providers = set()
for m in model_names:
    x_val  = metrics[m]["norm_accuracy"]
    y_val  = metrics[m]["norm_value"]
    speed  = metrics[m]["norm_speed"]
    prov   = metrics[m]["provider"]
    color  = PROVIDER_COLORS.get(prov, "#888888")
    size   = 300 + speed * 1200   # bubble size ∝ speed
    is_winner = (m == winner)

    ax.scatter(x_val, y_val, s=size, color=color,
               edgecolors="#FFD700" if is_winner else "white",
               linewidths=4 if is_winner else 1.5,
               zorder=5, alpha=0.92)

    # Label
    short = m.replace(" Instruct","").replace(" Maverick"," Mvk")
    offset_y = 0.07 if y_val < 0.85 else -0.07
    ax.annotate(
        ("⭐ " if is_winner else "") + short,
        xy=(x_val, y_val), xytext=(x_val, y_val + offset_y),
        ha="center", va="center", fontsize=8,
        fontweight="bold" if is_winner else "normal",
        color="#1a1a2e",
        bbox=dict(boxstyle="round,pad=0.25", fc="white", ec=color, alpha=0.85, lw=1.2)
    )

    # Data callout: acc / time / cost
    details = (
        f"acc={metrics[m]['avg_accuracy']:.2f}\n"
        f"{metrics[m]['avg_time_s']:.2f}s\n"
        f"${metrics[m]['total_cost']:.5f}"
    )
    ax.annotate(details, xy=(x_val, y_val),
                xytext=(x_val + 0.03, y_val - 0.11),
                fontsize=6.5, color="#555555", ha="left", va="top",
                bbox=dict(boxstyle="round,pad=0.2", fc="#F9F9F9", ec="#DDDDDD", alpha=0.8))
    used_providers.add(prov)

# Legend — provider colours
handles = [mpatches.Patch(color=PROVIDER_COLORS[p], label=p) for p in sorted(used_providers)]
handles.append(plt.scatter([], [], s=300, c="gray", label="Bubble size = Speed"))
ax.legend(handles=handles, fontsize=9, loc="lower right",
          framealpha=0.9, edgecolor="#CCCCCC")
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

# ── Right panel: Recommendation scorecard ────────────────────────────────────
ax2 = axes[1]
ax2.set_facecolor("#FFFFFF")
ax2.axis("off")
ax2.set_title("Recommendation Scorecard", fontsize=13, fontweight="bold", pad=10)

# Header
wm = metrics[winner]
ax2.text(0.5, 0.97,
    f"\n⭐  RECOMMENDED FOR\n"
    f"\"{USE_CASE_NAME}\"",
    ha="center", va="top", fontsize=11, fontweight="bold",
    color="#2e7d32", transform=ax2.transAxes, linespacing=1.6)

ax2.text(0.5, 0.80, winner,
    ha="center", va="top", fontsize=16, fontweight="bold",
    color=PROVIDER_COLORS.get(wm["provider"],"#333"),
    transform=ax2.transAxes)

ax2.add_patch(plt.Rectangle((0.05, 0.60), 0.90, 0.001,
    color="#CCCCCC", transform=ax2.transAxes))

# Metric rows
row_data = [
    ("Accuracy Score",   f"{wm['avg_accuracy']:.2f} / 1.00",    "#2e7d32"),
    ("Avg Response Time",f"{wm['avg_time_s']:.2f}s",             "#1565c0"),
    ("Total AI Cost",    f"${wm['total_cost']:.6f}",             "#6a1b9a"),
    ("Composite Score",  f"{wm['composite']:.3f} / 1.000",       "#e65100"),
    ("Provider",         wm["provider"],                          "#444444"),
    ("Profile Used",     PROFILE.replace("_"," ").title(),        "#444444"),
]
for i,(label,val,color) in enumerate(row_data):
    y = 0.55 - i*0.09
    ax2.text(0.08, y, label+":", ha="left", va="center", fontsize=9.5,
             color="#555555", transform=ax2.transAxes)
    ax2.text(0.92, y, val,      ha="right", va="center", fontsize=10,
             fontweight="bold", color=color, transform=ax2.transAxes)

# Runner-up
if len(ranked) > 1:
    runner = ranked[1]
    ax2.add_patch(plt.Rectangle((0.05, 0.01), 0.90, 0.001,
        color="#CCCCCC", transform=ax2.transAxes))
    ax2.text(0.5, 0.08,
        f"Runner-up: {runner}\n"
        f"Composite: {metrics[runner]['composite']:.3f}  "
        f"Accuracy: {metrics[runner]['avg_accuracy']:.2f}",
        ha="center", va="bottom", fontsize=8.5, color="#666666",
        transform=ax2.transAxes, linespacing=1.5)

plt.tight_layout(pad=2)
OUT = f"llm_magic_quadrant_{USE_CASE_NAME.lower().replace(' ','_')}.png"
plt.savefig(OUT, dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
print(f"\n✓ Magic Quadrant saved → {OUT}")
print(f"\n🏆 Recommendation: {winner}")
print(f"   Composite score : {wm['composite']:.3f}")
print(f"   Accuracy        : {wm['avg_accuracy']:.2f}")
print(f"   Avg time        : {wm['avg_time_s']:.2f}s")
print(f"   Total cost      : ${wm['total_cost']:.6f}")
plt.show()

---
## Cell 9 · Full scorecard table

In [ ]:
try:
    import pandas as pd
    rows = []
    for rank, m in enumerate(ranked, 1):
        x = metrics[m]
        def pct(val, ref): return f"{'+'if val>=ref else ''}{((val-ref)/ref*100):.1f}%" if ref else "N/A"
        mean_acc  = sum(metrics[n]["avg_accuracy"] for n in model_names)/len(model_names)
        mean_cost = sum(metrics[n]["total_cost"]   for n in model_names)/len(model_names)
        mean_time = sum(metrics[n]["avg_time_s"]   for n in model_names)/len(model_names)
        rows.append({
            "Rank":         f"#{rank}" + (" ⭐" if m==winner else ""),
            "Model":        m,
            "Provider":     x["provider"],
            "Accuracy":     round(x["avg_accuracy"],3),
            "Acc vs Mean":  pct(x["avg_accuracy"], mean_acc),
            "Avg Time (s)": round(x["avg_time_s"],2),
            "Time vs Mean": pct(x["avg_time_s"],   mean_time),
            "Total Cost $": round(x["total_cost"],6),
            "Cost vs Mean": pct(x["total_cost"],   mean_cost),
            "Composite":    round(x["composite"],3),
            "Errors":       x["errors"],
        })

    ACCURACY_COLORS = {"Excellent":"#c6efce","Good":"#ffeb9c","Partial":"#ffcc99","Poor":"#ffc7ce"}
    df = pd.DataFrame(rows)

    def color_acc(v):
        return "color:green;font-weight:bold" if isinstance(v,str) and v.startswith("+") else \
               "color:red;font-weight:bold"   if isinstance(v,str) and v.startswith("-") else ""
    def color_cost(v):
        return "color:red;font-weight:bold"   if isinstance(v,str) and v.startswith("+") else \
               "color:green;font-weight:bold" if isinstance(v,str) and v.startswith("-") else ""

    df.style \
        .background_gradient(subset=["Accuracy"],   cmap="RdYlGn") \
        .background_gradient(subset=["Total Cost $"],cmap="RdYlGn_r") \
        .background_gradient(subset=["Avg Time (s)"],cmap="RdYlGn_r") \
        .background_gradient(subset=["Composite"],   cmap="RdYlGn") \
        .applymap(color_acc,  subset=["Acc vs Mean"]) \
        .applymap(color_cost, subset=["Time vs Mean","Cost vs Mean"]) \
        .set_caption(f"Full scorecard — {USE_CASE_NAME}  |  Profile: {PROFILE}") \
        .hide(axis="index")
except ImportError:
    print("Install pandas for the styled table: %pip install pandas")

---
## Cell 10 · Export results

In [ ]:
safe_name = USE_CASE_NAME.lower().replace(" ","_")

# Raw results CSV
try:
    import pandas as pd
    csv_path = f"llm_benchmark_{safe_name}.csv"
    pd.DataFrame([{k:v for k,v in r.items() if k!="response"} for r in raw_results]) \
      .to_csv(csv_path, index=False)
    print(f"✓ Raw results  → {csv_path}")
except ImportError:
    pass

# Scorecard JSON
sc_path = f"llm_scorecard_{safe_name}.json"
with open(sc_path, "w") as f:
    json.dump({
        "use_case": USE_CASE_NAME,
        "profile":  PROFILE,
        "weights":  WEIGHTS,
        "recommendation": winner,
        "scores": {m: {
            "accuracy":   round(metrics[m]["avg_accuracy"],4),
            "avg_time_s": round(metrics[m]["avg_time_s"],3),
            "total_cost": round(metrics[m]["total_cost"],8),
            "composite":  round(metrics[m]["composite"],4),
        } for m in ranked}
    }, f, indent=2)
print(f"✓ Scorecard JSON → {sc_path}")
print(f"✓ Quadrant image → llm_magic_quadrant_{safe_name}.png")